# EDA: Ativos do Profit x Publicações de Trump sobre o conflito EUA-Irã

Análise exploratória relacionando o comportamento de WINFUT, WDOFUT e PETR4 (dados intradiários) com as publicações de Trump sobre o conflito.

Não assumimos que as publicações causaram os movimentos do mercado -- só estamos procurando padrões temporais.

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 2. Carregamento dos dados (Google Drive)

Montando o Drive e lendo cada arquivo separadamente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
CAMINHO = '/content/drive/MyDrive/excel_kj/'

df_winfut = pd.read_excel(CAMINHO + 'WINFUT1DIA.xlsx')
df_wdofut = pd.read_excel(CAMINHO + 'WDOFUT1DIA.xlsx')
df_petr4 = pd.read_excel(CAMINHO + 'PETR41DIA.xlsx')

df_winfut.head()


,Data,Abertura,Máxima,Mínima,Fechamento,Média Móvel A [21],Média Móvel A [200],Média Móvel A [9],VWAP D
0,2026-09-18,"187,905.00","188,730.00","185,880.00","187,135.00","183,460.24","187,133.10","188,503.89","186,688.19"
1,2026-09-17,"188,810.00","189,085.00","184,465.00","188,280.00","182,693.57","187,082.30","188,555.56","187,397.10"
2,2026-09-16,"189,320.00","189,730.00","186,410.00","187,600.00","181,805.95","187,028.75","188,490.56","187,661.19"
3,2026-09-15,"186,910.00","189,555.00","186,775.00","188,510.00","180,966.67","186,962.82","188,523.89","188,198.82"
4,2026-09-14,"186,900.00","188,260.00","185,040.00","187,250.00","180,084.05","186,890.55","187,843.89","186,727.38"


In [ ]:
df_wdofut.head()


,Data,Abertura,Máxima,Mínima,Fechamento,Média Móvel A [21],Média Móvel A [200],Média Móvel A [9],VWAP D
0,2026-09-18,"5,150.00","5,179.50","5,134.50","5,157.50","5,170.22","5,371.81","5,145.83","5,163.97"
1,2026-09-17,"5,153.00","5,193.00","5,139.00","5,140.00","5,173.70","5,374.77","5,145.67","5,164.77"
2,2026-09-16,"5,160.00","5,182.50","5,152.50","5,166.00","5,179.79","5,377.69","5,144.83","5,167.99"
3,2026-09-15,"5,171.00","5,180.50","5,145.00","5,171.00","5,184.25","5,380.75","5,140.50","5,165.27"
4,2026-09-14,"5,176.00","5,202.50","5,160.00","5,164.00","5,189.06","5,383.84","5,142.28","5,180.74"


In [ ]:
df_petr4.head()


,Data,Abertura,Máxima,Mínima,Fechamento,Média Móvel A [21],Média Móvel A [200],Média Móvel A [9],VWAP D
0,2026-09-18,48.40,48.83,48.25,48.47,46.28,39.28,48.86,48.51
1,2026-09-17,48.03,49.20,47.96,48.61,45.97,39.18,48.71,48.55
2,2026-09-16,49.94,49.98,48.62,48.65,45.62,39.09,48.59,49.00
3,2026-09-15,49.06,50.70,49.03,50.43,45.27,39.00,48.54,50.21
4,2026-09-14,49.51,49.73,48.61,48.92,44.82,38.89,48.14,49.20


## 3. Filtrar somente registros de 2026

Também ordeno por data aqui -- o arquivo do Profit vem com as datas fora de ordem, e isso é o que estava deixando os gráficos de candle invertidos (o candle mais recente aparecendo antes do mais antigo).

In [ ]:
df_winfut['Data'] = pd.to_datetime(df_winfut['Data'])
df_wdofut['Data'] = pd.to_datetime(df_wdofut['Data'])
df_petr4['Data'] = pd.to_datetime(df_petr4['Data'])

df_winfut = df_winfut[df_winfut['Data'].dt.year == 2026].sort_values('Data').reset_index(drop=True)
df_wdofut = df_wdofut[df_wdofut['Data'].dt.year == 2026].sort_values('Data').reset_index(drop=True)
df_petr4 = df_petr4[df_petr4['Data'].dt.year == 2026].sort_values('Data').reset_index(drop=True)

print('WINFUT:', df_winfut.shape)
print('WDOFUT:', df_wdofut.shape)
print('PETR4:', df_petr4.shape)


WINFUT: (179, 9)
WDOFUT: (179, 9)
PETR4: (179, 9)


In [ ]:
df_winfut.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179 entries, 0 to 178
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Data                 179 non-null    datetime64[ns]
 1   Abertura             179 non-null    float64       
 2   Máxima               179 non-null    float64       
 3   Mínima               179 non-null    float64       
 4   Fechamento           179 non-null    float64       
 5   Média Móvel A [21]   179 non-null    float64       
 6   Média Móvel A [200]  179 non-null    float64       
 7   Média Móvel A [9]    179 non-null    float64       
 8   VWAP D               179 non-null    float64       
dtypes: datetime64[ns](1), float64(8)
memory usage: 12.7 KB


In [ ]:
df_winfut.describe()


,Data,Abertura,Máxima,Mínima,Fechamento,Média Móvel A [21],Média Móvel A [200],Média Móvel A [9],VWAP D
count,179,179.00,179.00,179.00,179.00,179.00,179.00,179.00,179.00
mean,2026-05-12 20:38:52.960893952,"188,331.54","190,017.96","186,670.41","188,302.24","187,743.54","178,008.61","188,047.82","188,277.04"
min,2026-01-02 00:00:00,"169,635.00","171,170.00","167,975.00","169,640.00","175,321.75","164,077.54","170,956.11","169,159.53"
25%,2026-03-09 12:00:00,"178,864.41","180,454.85","177,576.11","178,712.35","178,406.27","171,870.52","178,830.29","178,917.48"
50%,2026-05-14 00:00:00,"186,285.65","187,695.80","184,465.00","186,660.30","183,460.24","180,103.70","185,515.53","185,852.02"
75%,2026-07-16 12:00:00,"197,713.27","200,072.27","196,004.78","197,724.95","197,885.62","184,495.96","198,071.24","197,969.51"
max,2026-09-18 00:00:00,"211,157.33","212,129.92","210,572.72","211,077.61","204,085.96","187,133.10","208,958.64","211,312.09"
std,NaN,"10,997.74","11,091.57","10,795.70","10,940.37","10,128.75","7,323.46","10,687.43","10,899.63"


## 4. Publicações de Trump sobre o conflito

Preencher abaixo com as publicações reais (data e texto). Estrutura de exemplo:

In [ ]:
publicacoes_trump = pd.DataFrame({
    'data': [
        '2026-01-13',
        '2026-02-06',
        '2026-02-28'
    ],
    'texto': [
        'Trump publicou no Truth Social uma mensagem incentivando os iranianos a continuarem os protestos e dizendo que “help is on its way”',
        'Começaram negociações nucleares indiretas EUA–Irã em Genebra. No mesmo dia, Trump falou sobre as negociações e mencionou a presença de uma grande frota americana na região.',
        'EUA e Israel iniciaram ataques contra o Irã. Trump anunciou que os EUA haviam iniciado “major combat operations”'
    ],
})

publicacoes_trump['data'] = pd.to_datetime(publicacoes_trump['data'])
publicacoes_trump


,data,texto
0,2026-01-13,Trump publicou no Truth Social uma mensagem in...
1,2026-02-06,Começaram negociações nucleares indiretas EUA–...
2,2026-02-28,EUA e Israel iniciaram ataques contra o Irã. T...


## 5. Comportamento dos ativos antes e depois das publicações

Para cada publicação, olhamos o fechamento do dia anterior, do dia da publicação e do dia seguinte.

In [ ]:
def fechamento_ao_redor(df, data_pub, dias=1):
    data_pub = pd.to_datetime(data_pub)
    inicio = data_pub - pd.Timedelta(days=dias)
    fim = data_pub + pd.Timedelta(days=dias)
    return df[(df['Data'] >= inicio) & (df['Data'] <= fim)][['Data', 'Fechamento']]


In [ ]:
for _, pub in publicacoes_trump.iterrows():
    print('Publicação:', pub['data'].date(), '-', pub['texto'])
    print('WINFUT:')
    print(fechamento_ao_redor(df_winfut, pub['data']))
    print('WDOFUT:')
    print(fechamento_ao_redor(df_wdofut, pub['data']))
    print('PETR4:')
    print(fechamento_ao_redor(df_petr4, pub['data']))
    print('---')


Publicação: 2026-01-13 - Trump publicou no Truth Social uma mensagem incentivando os iranianos a continuarem os protestos e dizendo que “help is on its way”
WINFUT:
        Data  Fechamento
6 2026-01-12  179,273.44
7 2026-01-13  177,636.34
8 2026-01-14  181,338.79
WDOFUT:
        Data  Fechamento
6 2026-01-12    5,712.38
7 2026-01-13    5,706.56
8 2026-01-14    5,730.88
PETR4:
        Data  Fechamento
6 2026-01-12       28.77
7 2026-01-13       29.51
8 2026-01-14       30.31
---
Publicação: 2026-02-06 - Começaram negociações nucleares indiretas EUA–Irã em Genebra. No mesmo dia, Trump falou sobre as negociações e mencionou a presença de uma grande frota americana na região.
WINFUT:
         Data  Fechamento
24 2026-02-05  198,382.00
25 2026-02-06  198,815.66
WDOFUT:
         Data  Fechamento
24 2026-02-05    5,568.62
25 2026-02-06    5,507.66
PETR4:
         Data  Fechamento
24 2026-02-05       35.06
25 2026-02-06       34.73
---
Publicação: 2026-02-28 - EUA e Israel iniciaram ataques c

In [ ]:
def retorno_pre_pos(df, data_pub, dias=1):
    data_pub = pd.to_datetime(data_pub)
    antes = df[df['Data'] < data_pub].tail(dias)
    depois = df[df['Data'] > data_pub].head(dias)
    if antes.empty or depois.empty:
        return np.nan
    fechamento_antes = antes['Fechamento'].iloc[-1]
    fechamento_depois = depois['Fechamento'].iloc[-1]
    return (fechamento_depois - fechamento_antes) / fechamento_antes


resultados = []
for _, pub in publicacoes_trump.iterrows():
    resultados.append({
        'data_publicacao': pub['data'],
        'retorno_winfut': retorno_pre_pos(df_winfut, pub['data']),
        'retorno_wdofut': retorno_pre_pos(df_wdofut, pub['data']),
        'retorno_petr4': retorno_pre_pos(df_petr4, pub['data']),
    })

df_resultados = pd.DataFrame(resultados)
df_resultados


,data_publicacao,retorno_winfut,retorno_wdofut,retorno_petr4
0,2026-01-13,0.01,0.00,0.05
1,2026-02-06,0.02,-0.02,0.01
2,2026-02-28,0.00,0.01,0.05


## 6. Gráficos ao redor de cada ocorrido

Duas funções: uma de candles (serve tanto pra janela de um ocorrido quanto pro período completo) e uma de linha com bandeira marcando os ocorridos. No final da seção tem um bloco só com chamadas comentadas -- descomente só o que você quiser ver.

In [ ]:
JANELA_DIAS = 14
LIMITE_TEXTO_TITULO = 40


def truncar_texto(texto, limite=LIMITE_TEXTO_TITULO):
    texto = str(texto)
    if len(texto) > limite:
        return texto[:limite].rstrip() + '...'
    return texto


def recortar_janela(df, data_evento, dias=JANELA_DIAS):
    inicio = data_evento - pd.Timedelta(days=dias)
    fim = data_evento + pd.Timedelta(days=dias)
    return df[(df['Data'] >= inicio) & (df['Data'] <= fim)].reset_index(drop=True)


### 6.1 Candles

Se passar `data_evento`, mostra só a janela de 14 dias antes/depois dessa data (com linha tracejada marcando o dia). Se não passar nada, mostra o período completo do ativo.

In [ ]:
from matplotlib.patches import Rectangle

def plotar_candles(df, titulo, data_evento=None, dias=JANELA_DIAS):
    if data_evento is not None:
        dados = recortar_janela(df, data_evento, dias)
    else:
        dados = df.reset_index(drop=True)

    if dados.empty:
        print('Sem dados para', titulo)
        return

    fig, ax = plt.subplots(figsize=(14, 6))

    for i, linha in dados.iterrows():
        cor = 'green' if linha['Fechamento'] >= linha['Abertura'] else 'red'
        ax.plot([i, i], [linha['Mínima'], linha['Máxima']], color=cor, linewidth=1)
        ax.add_patch(Rectangle(
            (i - 0.3, min(linha['Abertura'], linha['Fechamento'])),
            0.6,
            max(abs(linha['Fechamento'] - linha['Abertura']), 0.01),
            color=cor,
        ))

    if data_evento is not None:
        idx_evento = (dados['Data'] - data_evento).abs().idxmin()
        ax.axvline(idx_evento, color='blue', linestyle='--', alpha=0.7)

    passo = max(1, len(dados) // 20)
    ax.set_xticks(range(0, len(dados), passo))
    ax.set_xticklabels(dados['Data'].dt.strftime('%d/%m/%y').iloc[::passo], rotation=90, fontsize=8)
    ax.set_title(truncar_texto(titulo, 70))
    plt.tight_layout()
    plt.show()


### 6.2 Ícone (bandeira) a partir de um arquivo

Em vez de desenhar a bandeira, agora carrego um ícone de um arquivo de imagem salvo na raiz do ambiente. No Colab, isso normalmente é `/content/nome_do_arquivo.png` -- ajuste `CAMINHO_ICONE` abaixo pro nome do seu arquivo.

In [ ]:
from PIL import Image
import base64

CAMINHO_ICONE = 'icons8-donald-trump-64.png'  # coloque o arquivo na raiz do ambiente e ajuste o nome aqui


def carregar_icone_matplotlib(caminho=CAMINHO_ICONE, transparencia=1.0):
    imagem = np.array(Image.open(caminho).convert('RGBA')).astype(np.float64)
    imagem[..., 3] = np.clip(imagem[..., 3] * transparencia, 0, 255)
    return imagem.astype(np.uint8)


def carregar_icone_base64(caminho=CAMINHO_ICONE):
    with open(caminho, 'rb') as arquivo:
        codificado = base64.b64encode(arquivo.read()).decode()
    extensao = caminho.split('.')[-1].lower()
    return f'data:image/{extensao};base64,{codificado}'


icone_bandeira = carregar_icone_matplotlib()
icone_base64 = carregar_icone_base64()


### 6.3 Linha estática com o ícone marcando os ocorridos

In [ ]:
def plotar_linha_com_bandeiras(df, titulo, dias=JANELA_DIAS):
    data_min = publicacoes_trump['data'].min() - pd.Timedelta(days=dias)
    data_max = publicacoes_trump['data'].max() + pd.Timedelta(days=dias)
    dados = df[(df['Data'] >= data_min) & (df['Data'] <= data_max)].reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(16, 6))
    ax.plot(dados['Data'], dados['Fechamento'], color='steelblue', linewidth=1.5, zorder=2)

    for _, pub in publicacoes_trump.iterrows():
        idx_mais_proximo = (dados['Data'] - pub['data']).abs().idxmin()
        x = dados.loc[idx_mais_proximo, 'Data']
        y = dados.loc[idx_mais_proximo, 'Fechamento']
        caixa_imagem = OffsetImage(icone_bandeira, zoom=0.5)
        anotacao = AnnotationBbox(caixa_imagem, (x, y), frameon=False, box_alignment=(0.5, 0), zorder=1)
        ax.add_artist(anotacao)

    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
    plt.setp(ax.get_xticklabels(), rotation=90, fontsize=7)

    ax.set_title(truncar_texto(titulo, 70))
    ax.set_xlabel('Data')
    ax.set_ylabel('Fechamento')
    plt.tight_layout()
    plt.show()


### 6.4 Versão interativa (com mouse)

O matplotlib normal não faz hover -- pra ter a linha seguindo o mouse e uma caixinha com os valores do dia (data, abertura, máxima, mínima, fechamento e o ocorrido, se houver), uso o Plotly aqui. Já roda direto no Colab, sem instalar nada.

In [ ]:
import plotly.graph_objects as go


def plotar_linha_interativa(df, titulo, dias=JANELA_DIAS):
    data_min = publicacoes_trump['data'].min() - pd.Timedelta(days=dias)
    data_max = publicacoes_trump['data'].max() + pd.Timedelta(days=dias)
    dados = df[(df['Data'] >= data_min) & (df['Data'] <= data_max)].reset_index(drop=True)

    # marca o texto do ocorrido só no dia mais próximo dele
    dados['ocorrido'] = ''
    for _, pub in publicacoes_trump.iterrows():
        idx = (dados['Data'] - pub['data']).abs().idxmin()
        dados.loc[idx, 'ocorrido'] = truncar_texto(pub['texto'], 60)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=dados['Data'],
        y=dados['Fechamento'],
        mode='lines',
        line=dict(color='steelblue', width=2),
        customdata=dados[['Abertura', 'Máxima', 'Mínima', 'Fechamento', 'ocorrido']],
        hovertemplate=(
            '<b>%{x|%d/%m/%Y}</b><br>'
            'Abertura: %{customdata[0]:.2f}<br>'
            'Máxima: %{customdata[1]:.2f}<br>'
            'Mínima: %{customdata[2]:.2f}<br>'
            'Fechamento: %{customdata[3]:.2f}<br>'
            '%{customdata[4]}<extra></extra>'
        ),
    ))

    # bandeira semitransparente em cima de cada dia com ocorrido
    variacao_preco = dados['Fechamento'].max() - dados['Fechamento'].min()
    for _, linha in dados[dados['ocorrido'] != ''].iterrows():
        fig.add_layout_image(dict(
            source=icone_base64,
            x=linha['Data'],
            y=linha['Fechamento'],
            xref='x', yref='y',
            sizex=2 * 24 * 60 * 60 * 1000,  # ~2 dias, em milissegundos (eixo de data)
            sizey=variacao_preco * 0.12,
            xanchor='center', yanchor='bottom',
            opacity=1.0,
            layer='above',
        ))

    # linha vertical acompanhando o mouse (crosshair)
    fig.update_xaxes(showspikes=True, spikemode='across', spikesnap='cursor',
                      spikecolor='gray', spikethickness=1, spikedash='solid')
    fig.update_yaxes(showspikes=True, spikecolor='gray', spikethickness=1)

    fig.update_layout(
        title=truncar_texto(titulo, 70),
        xaxis_title='Data',
        yaxis_title='Fechamento',
        hovermode='x',
        hoverlabel=dict(bgcolor='rgba(255,255,255,0.75)', font_size=12),
        height=550,
        width=1000,
    )

    fig.show()


### 6.5 Candles interativo (todos os ocorridos juntos)

Mesmo padrão do gráfico de linha interativo (6.4) -- crosshair, hover com data/OHLC/ocorrido, ícone marcando cada data -- só que em candle em vez de linha.

In [ ]:
def plotar_candles_interativo(df, titulo, dias=JANELA_DIAS):
    data_min = publicacoes_trump['data'].min() - pd.Timedelta(days=dias)
    data_max = publicacoes_trump['data'].max() + pd.Timedelta(days=dias)
    dados = df[(df['Data'] >= data_min) & (df['Data'] <= data_max)].reset_index(drop=True)

    dados['ocorrido'] = ''
    for _, pub in publicacoes_trump.iterrows():
        idx = (dados['Data'] - pub['data']).abs().idxmin()
        dados.loc[idx, 'ocorrido'] = truncar_texto(pub['texto'], 60)

    fig = go.Figure()

    # candle sem hover próprio (hoverinfo='skip') -- quem mostra a caixinha é a
    # linha invisível logo abaixo. Isso evita a duplicação de Abertura/Máxima/etc.
    fig.add_trace(go.Candlestick(
        x=dados['Data'],
        open=dados['Abertura'],
        high=dados['Máxima'],
        low=dados['Mínima'],
        close=dados['Fechamento'],
        increasing_line_color='green',
        decreasing_line_color='red',
        hoverinfo='skip',
        name='',
    ))

    # marcadores invisíveis só pra controlar o hover
    fig.add_trace(go.Scatter(
        x=dados['Data'],
        y=dados['Fechamento'],
        mode='markers',
        marker=dict(opacity=0),
        customdata=dados[['Abertura', 'Máxima', 'Mínima', 'Fechamento', 'ocorrido']],
        hovertemplate=(
            '<b>%{x|%d/%m/%Y}</b><br>'
            'Abertura: %{customdata[0]:.2f}<br>'
            'Máxima: %{customdata[1]:.2f}<br>'
            'Mínima: %{customdata[2]:.2f}<br>'
            'Fechamento: %{customdata[3]:.2f}<br>'
            '%{customdata[4]}<extra></extra>'
        ),
        showlegend=False,
    ))

    variacao_preco = dados['Máxima'].max() - dados['Mínima'].min()

    # ALTERE AQUI o tamanho do ícone:
    # TAMANHO_ICONE_X é a largura, em milissegundos (eixo é de datas). 1 dia = 86400000.
    # TAMANHO_ICONE_Y é a altura, em proporção da variação de preço do período exibido.
    TAMANHO_ICONE_X = 2 * 24 * 60 * 60 * 1000
    TAMANHO_ICONE_Y = variacao_preco * 0.12

    for _, linha in dados[dados['ocorrido'] != ''].iterrows():
        fig.add_layout_image(dict(
            source=icone_base64,
            x=linha['Data'],
            y=linha['Máxima'],
            xref='x', yref='y',
            sizex=TAMANHO_ICONE_X,
            sizey=TAMANHO_ICONE_Y,
            xanchor='center', yanchor='bottom',
            opacity=1.0,
            layer='above',
        ))

    todas_as_datas = pd.date_range(dados['Data'].min(), dados['Data'].max(), freq='D')
    dias_sem_dado = [d for d in todas_as_datas if d not in set(dados['Data'])]

    fig.update_xaxes(
        showspikes=True, spikemode='across', spikesnap='cursor',
        spikecolor='gray', spikethickness=1, spikedash='solid',
        gridcolor='white',
        rangebreaks=[dict(values=dias_sem_dado)],
    )
    fig.update_yaxes(showspikes=True, spikecolor='gray', spikethickness=1, gridcolor='white')

    fig.update_layout(
        title=truncar_texto(titulo, 70),
        xaxis_title='Data',
        yaxis_title='Preço',
        xaxis_rangeslider_visible=False,
        hovermode='x',
        hoverlabel=dict(bgcolor='rgba(255,255,255,0.75)', font_size=12),
        plot_bgcolor='#d9d9d9',
        paper_bgcolor='#d9d9d9',
        height=550,
        width=1000,
    )

    fig.show()


### 6.6 exibindo resultados..

In [ ]:
# --- Candles por ocorrido (janela de 14 dias antes/depois) ---
# for _, pub in publicacoes_trump.iterrows():
#     plotar_candles(df_winfut, f"WINFUT - {pub['data'].date()} - {pub['texto']}", data_evento=pub['data'])

# for _, pub in publicacoes_trump.iterrows():
#     plotar_candles(df_wdofut, f"WDOFUT - {pub['data'].date()} - {pub['texto']}", data_evento=pub['data'])

# for _, pub in publicacoes_trump.iterrows():
#     plotar_candles(df_petr4, f"PETR4 - {pub['data'].date()} - {pub['texto']}", data_evento=pub['data'])


In [ ]:


# --- Candles do período completo ---
# plotar_candles(df_winfut, 'WINFUT - período completo')
# plotar_candles(df_wdofut, 'WDOFUT - período completo')
# plotar_candles(df_petr4, 'PETR4 - período completo')


# --- Linha estática com bandeiras marcando os ocorridos ---
# plotar_linha_com_bandeiras(df_winfut, 'WINFUT - Fechamento com ocorridos marcados')
# plotar_linha_com_bandeiras(df_wdofut, 'WDOFUT - Fechamento com ocorridos marcados')
# plotar_linha_com_bandeiras(df_petr4, 'PETR4 - Fechamento com ocorridos marcados')


# --- Linha interativa (hover com mouse) ---
# plotar_linha_interativa(df_winfut, 'WINFUT - Fechamento com ocorridos marcados')
# plotar_linha_interativa(df_wdofut, 'WDOFUT - Fechamento com ocorridos marcados')
plotar_linha_interativa(df_petr4, 'PETR4 - Fechamento com ocorridos marcados')


# --- Candles interativo (hover com mouse, todos os ocorridos juntos) ---
# plotar_candles_interativo(df_winfut, 'WINFUT - Candles com ocorridos marcados')
# plotar_candles_interativo(df_wdofut, 'WDOFUT - Candles com ocorridos marcados')
plotar_candles_interativo(df_petr4, 'PETR4 - Candles com ocorridos marcados')


## 7. Observações

Preencher após rodar com os dados reais e as publicações completas.

- O que aconteceu com cada ativo antes/depois de cada publicação?
- Os movimentos foram parecidos entre os três ativos ou diferentes?
- Isso não prova que a publicação causou o movimento -- pode ser coincidência ou outro fator do mesmo dia.